# Entity Identification Pipeline for Co-Pilot Agent

## Overview
This notebook implements three approaches for predicting relevant entities from user queries:

1. **Naive LLM Approach**: Full context in-context learning - provide ALL training data + entity descriptions to LLM
2. **Fine-tuned Classifier**: Train a DistilBERT model for multi-label classification
3. **RAG Approach**: Retrieve most relevant examples + entity descriptions for each query

## Entity Types
- CDR (Call Detail Records)
- Phone
- Web Activity
- Web Actor
- Person
- Investigation
- Insight
- Report
- EVisa Request

## 1. Setup and Dependencies

In [ ]:
# Install required packages
# !pip install pandas numpy scikit-learn torch transformers sentence-transformers accelerate

import pandas as pd
import numpy as np
import json
import ast
import re
from typing import List, Dict, Set, Tuple, Optional
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# ML imports
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, hamming_loss, jaccard_score
)

# Deep learning imports
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    AutoModelForCausalLM, TrainingArguments, Trainer
)

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

## 2. Load and Preprocess Data

In [ ]:
# Load datasets
user_queries_df = pd.read_csv('user_queries.csv')
fields_description_df = pd.read_csv('fields_description.csv')

print(f"User Queries: {len(user_queries_df)} rows")
print(f"Fields Description: {len(fields_description_df)} rows")
print(f"\nEntity types in fields_description: {fields_description_df['entity_name'].unique().tolist()}")

In [ ]:
def safe_parse_json(json_str: str) -> dict:
    """Parse JSON string, handling Python dict format."""
    try:
        return json.loads(json_str)
    except json.JSONDecodeError:
        try:
            return ast.literal_eval(json_str)
        except (ValueError, SyntaxError):
            return {}

def extract_entities(json_obj: dict) -> List[str]:
    """Extract entities from entityType and relationTargetType keys."""
    entities = set()
    
    if 'entityType' in json_obj:
        entities.add(json_obj['entityType'])
    
    def search_statements(statements):
        if not statements:
            return
        for stmt in statements:
            if isinstance(stmt, dict):
                params = stmt.get('parameters', {})
                if 'relationTargetType' in params:
                    targets = params['relationTargetType']
                    if isinstance(targets, list):
                        entities.update(targets)
                    else:
                        entities.add(targets)
                if 'statements' in stmt:
                    search_statements(stmt['statements'])
    
    if 'statements' in json_obj:
        search_statements(json_obj['statements'])
    
    return sorted(list(entities))

# Parse and extract entities
user_queries_df['parsed_json'] = user_queries_df['json'].apply(safe_parse_json)
user_queries_df['entities'] = user_queries_df['parsed_json'].apply(extract_entities)

# Show distribution
all_entities = [e for ents in user_queries_df['entities'] for e in ents]
entity_counts = Counter(all_entities)
print("Entity Distribution:")
for entity, count in entity_counts.most_common():
    print(f"  {entity}: {count} ({100*count/len(user_queries_df):.1f}%)")

multi_entity = sum(1 for e in user_queries_df['entities'] if len(e) > 1)
print(f"\nQueries with multiple entities: {multi_entity} ({100*multi_entity/len(user_queries_df):.1f}%)")

In [ ]:
# Build entity descriptions from fields_description.csv
def build_entity_descriptions(fields_df: pd.DataFrame) -> str:
    """Build a comprehensive description of all entities and their fields."""
    descriptions = []
    
    for entity in sorted(fields_df['entity_name'].unique()):
        entity_fields = fields_df[fields_df['entity_name'] == entity]
        desc_lines = [f"\n## {entity}"]
        
        for _, row in entity_fields.head(8).iterrows():  # Limit fields per entity
            field_name = row['field_name'].split('.')[-1]
            field_desc = str(row['description'])[:100] if pd.notna(row['description']) else ''
            desc_lines.append(f"  - {field_name}: {field_desc}")
        
        descriptions.append('\n'.join(desc_lines))
    
    return '\n'.join(descriptions)

ENTITY_DESCRIPTIONS = build_entity_descriptions(fields_description_df)
print("Entity Descriptions (truncated):")
print(ENTITY_DESCRIPTIONS[:2000] + "...")

In [ ]:
# Prepare train/test split
ALL_ENTITIES = sorted(list(set(all_entities)))
print(f"All entity types ({len(ALL_ENTITIES)}): {ALL_ENTITIES}")

mlb = MultiLabelBinarizer(classes=ALL_ENTITIES)
y_encoded = mlb.fit_transform(user_queries_df['entities'])

# Split: 80% train, 20% test
X = user_queries_df['question'].tolist()
y = y_encoded
indices = list(range(len(X)))

train_idx, test_idx, y_train, y_test = train_test_split(
    indices, y, test_size=0.2, random_state=42
)

X_train = [X[i] for i in train_idx]
X_test = [X[i] for i in test_idx]
train_entities = [user_queries_df.iloc[i]['entities'] for i in train_idx]
test_entities = [user_queries_df.iloc[i]['entities'] for i in test_idx]

# Also keep full rows for context building
train_df = user_queries_df.iloc[train_idx].reset_index(drop=True)
test_df = user_queries_df.iloc[test_idx].reset_index(drop=True)

print(f"\nTrain size: {len(X_train)}")
print(f"Test size: {len(X_test)}")

## 3. Evaluation Framework

In [ ]:
def evaluate_predictions(y_true: np.ndarray, y_pred: np.ndarray, 
                        label_names: List[str], method_name: str = "Method") -> Dict:
    """Compute and display evaluation metrics."""
    metrics = {
        'exact_match': accuracy_score(y_true, y_pred),
        'hamming_loss': hamming_loss(y_true, y_pred),
        'f1_micro': f1_score(y_true, y_pred, average='micro', zero_division=0),
        'f1_macro': f1_score(y_true, y_pred, average='macro', zero_division=0),
        'precision_micro': precision_score(y_true, y_pred, average='micro', zero_division=0),
        'recall_micro': recall_score(y_true, y_pred, average='micro', zero_division=0),
        'jaccard_micro': jaccard_score(y_true, y_pred, average='micro', zero_division=0),
    }
    
    print(f"\n{'='*60}")
    print(f"RESULTS: {method_name}")
    print(f"{'='*60}")
    print(f"Exact Match Accuracy: {metrics['exact_match']:.4f}")
    print(f"F1 Micro:             {metrics['f1_micro']:.4f}")
    print(f"F1 Macro:             {metrics['f1_macro']:.4f}")
    print(f"Precision Micro:      {metrics['precision_micro']:.4f}")
    print(f"Recall Micro:         {metrics['recall_micro']:.4f}")
    print(f"Jaccard Micro:        {metrics['jaccard_micro']:.4f}")
    print(f"Hamming Loss:         {metrics['hamming_loss']:.4f}")
    print(f"\nPer-Class Report:")
    print(classification_report(y_true, y_pred, target_names=label_names, zero_division=0))
    
    return metrics

def parse_llm_response(response: str, valid_entities: List[str]) -> List[str]:
    """Parse LLM response to extract entity names."""
    predicted = []
    response_lower = response.lower()
    
    for entity in valid_entities:
        # Check for exact match or common variations
        if entity.lower() in response_lower:
            predicted.append(entity)
        # Handle special cases
        elif entity == 'EVisa Request' and ('visa' in response_lower or 'evisa' in response_lower):
            predicted.append(entity)
        elif entity == 'Web Activity' and 'webactivity' in response_lower.replace(' ', ''):
            predicted.append(entity)
        elif entity == 'Web Actor' and 'webactor' in response_lower.replace(' ', ''):
            predicted.append(entity)
    
    return predicted if predicted else ['CDR']  # Default fallback

---
# APPROACH 1: Naive LLM (Full Context)

This approach provides the LLM with:
- Full entity descriptions from `fields_description.csv`
- ALL training examples (query -> entities) as context
- The test query to predict

The LLM uses in-context learning with maximum available information.

In [ ]:
class NaiveLLMPredictor:
    """
    Naive LLM approach: Give ALL training data as context.
    Uses a small local LLM (TinyLlama, Phi-2, or similar).
    """
    
    def __init__(self, model_name: str = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"):
        self.model_name = model_name
        self.device = DEVICE
        self.model = None
        self.tokenizer = None
        self.context = None
        self.max_context_tokens = 3500  # Leave room for query and response
        
    def load_model(self):
        """Load the LLM model."""
        print(f"Loading {self.model_name}...")
        self.tokenizer = AutoTokenizer.from_pretrained(self.model_name)
        
        if self.device == "cuda":
            self.model = AutoModelForCausalLM.from_pretrained(
                self.model_name, torch_dtype=torch.float16, device_map="auto"
            )
        else:
            self.model = AutoModelForCausalLM.from_pretrained(
                self.model_name, torch_dtype=torch.float32, low_cpu_mem_usage=True
            )
            self.model.to(self.device)
        
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token
        
        print("Model loaded!")
    
    def build_full_context(self, train_df: pd.DataFrame, entity_descriptions: str,
                          all_entities: List[str]):
        """
        Build the full context with ALL training examples.
        Truncates if necessary to fit context window.
        """
        # System prompt with entity info
        system_prompt = f"""You are an entity extraction system for a co-pilot agent.
Given a user query, predict which entity types are relevant.

AVAILABLE ENTITY TYPES: {', '.join(all_entities)}

ENTITY DESCRIPTIONS:
{entity_descriptions}

TRAINING EXAMPLES:
"""
        
        # Add as many training examples as fit
        examples = []
        for _, row in train_df.iterrows():
            example = f"Query: {row['question']}\nEntities: {row['entities']}\n"
            examples.append(example)
        
        # Build context and check token count
        full_context = system_prompt + "\n".join(examples)
        
        # Truncate if needed
        tokens = self.tokenizer.encode(full_context)
        if len(tokens) > self.max_context_tokens:
            print(f"Context too long ({len(tokens)} tokens), truncating...")
            # Keep system prompt and as many examples as fit
            system_tokens = self.tokenizer.encode(system_prompt)
            available_tokens = self.max_context_tokens - len(system_tokens) - 100
            
            truncated_examples = []
            current_tokens = 0
            for ex in examples:
                ex_tokens = len(self.tokenizer.encode(ex))
                if current_tokens + ex_tokens < available_tokens:
                    truncated_examples.append(ex)
                    current_tokens += ex_tokens
                else:
                    break
            
            full_context = system_prompt + "\n".join(truncated_examples)
            print(f"Kept {len(truncated_examples)}/{len(examples)} examples")
        
        self.context = full_context
        print(f"Context built: {len(self.tokenizer.encode(self.context))} tokens")
    
    def predict(self, query: str) -> List[str]:
        """Predict entities for a query using full context."""
        prompt = f"""{self.context}

Now predict the entities for this query:
Query: {query}
Entities:"""
        
        inputs = self.tokenizer(prompt, return_tensors="pt", truncation=True, 
                               max_length=4096)
        inputs = {k: v.to(self.device) for k, v in inputs.items()}
        
        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=100,
                do_sample=False,
                pad_token_id=self.tokenizer.pad_token_id,
                eos_token_id=self.tokenizer.eos_token_id,
                temperature=0.1
            )
        
        response = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
        
        # Extract the part after "Entities:"
        if "Entities:" in response:
            response = response.split("Entities:")[-1].strip()
        
        return parse_llm_response(response, ALL_ENTITIES)
    
    def predict_batch(self, queries: List[str], verbose: bool = True) -> List[List[str]]:
        """Predict entities for multiple queries."""
        predictions = []
        for i, query in enumerate(queries):
            if verbose and i % 20 == 0:
                print(f"Processing {i+1}/{len(queries)}...")
            pred = self.predict(query)
            predictions.append(pred)
        return predictions

In [ ]:
# Initialize and run Naive LLM approach
print("=" * 60)
print("APPROACH 1: NAIVE LLM (Full Context)")
print("=" * 60)

naive_predictor = NaiveLLMPredictor(model_name="TinyLlama/TinyLlama-1.1B-Chat-v1.0")
naive_predictor.load_model()
naive_predictor.build_full_context(train_df, ENTITY_DESCRIPTIONS, ALL_ENTITIES)

In [ ]:
# Run predictions on test set
print("\nRunning Naive LLM predictions on test set...")
naive_predictions = naive_predictor.predict_batch(X_test)

# Encode predictions for evaluation
naive_pred_encoded = mlb.transform(naive_predictions)

# Evaluate
naive_metrics = evaluate_predictions(y_test, naive_pred_encoded, ALL_ENTITIES, 
                                     "Naive LLM (Full Context)")

---
# APPROACH 2: Fine-tuned Classifier (DistilBERT)

This approach:
- Fine-tunes a DistilBERT model for multi-label classification
- Trains on the training set
- Fast inference, no context needed at prediction time

In [ ]:
class EntityDataset(Dataset):
    """PyTorch Dataset for entity classification."""
    
    def __init__(self, texts: List[str], labels: np.ndarray, tokenizer, max_length: int = 128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(self.labels[idx], dtype=torch.float)
        }


class FineTunedClassifier:
    """Fine-tuned DistilBERT for multi-label entity classification."""
    
    def __init__(self, model_name: str = "distilbert-base-uncased", num_labels: int = 9):
        self.model_name = model_name
        self.num_labels = num_labels
        self.device = DEVICE
        self.model = None
        self.tokenizer = None
        self.mlb = None
    
    def train(self, X_train: List[str], y_train: np.ndarray, 
              X_val: List[str] = None, y_val: np.ndarray = None,
              epochs: int = 10, batch_size: int = 16):
        """Train the classifier."""
        print(f"Initializing {self.model_name}...")
        
        self.tokenizer = AutoTokenizer.from_pretrained(self.model_name)
        self.model = AutoModelForSequenceClassification.from_pretrained(
            self.model_name,
            num_labels=self.num_labels,
            problem_type="multi_label_classification"
        )
        
        # Create datasets
        train_dataset = EntityDataset(X_train, y_train, self.tokenizer)
        
        # Split train into train/val if not provided
        if X_val is None:
            X_t, X_v, y_t, y_v = train_test_split(X_train, y_train, test_size=0.15, random_state=42)
            train_dataset = EntityDataset(X_t, y_t, self.tokenizer)
            val_dataset = EntityDataset(X_v, y_v, self.tokenizer)
        else:
            val_dataset = EntityDataset(X_val, y_val, self.tokenizer)
        
        # Training arguments
        training_args = TrainingArguments(
            output_dir='./classifier_output',
            num_train_epochs=epochs,
            per_device_train_batch_size=batch_size,
            per_device_eval_batch_size=batch_size,
            warmup_steps=100,
            weight_decay=0.01,
            logging_steps=50,
            eval_strategy='epoch',
            save_strategy='epoch',
            load_best_model_at_end=True,
            metric_for_best_model='eval_loss',
            report_to='none',
            fp16=torch.cuda.is_available(),
        )
        
        # Train
        trainer = Trainer(
            model=self.model,
            args=training_args,
            train_dataset=train_dataset,
            eval_dataset=val_dataset,
        )
        
        print("Training...")
        trainer.train()
        print("Training complete!")
        
        self.model.eval()
    
    def predict(self, query: str, threshold: float = 0.5) -> List[str]:
        """Predict entities for a single query."""
        inputs = self.tokenizer(
            query, return_tensors='pt', truncation=True, 
            padding=True, max_length=128
        )
        inputs = {k: v.to(self.device) for k, v in inputs.items()}
        
        self.model.to(self.device)
        
        with torch.no_grad():
            outputs = self.model(**inputs)
            probs = torch.sigmoid(outputs.logits).cpu().numpy()[0]
        
        predicted = [ALL_ENTITIES[i] for i, p in enumerate(probs) if p > threshold]
        return predicted if predicted else ['CDR']
    
    def predict_batch(self, queries: List[str], threshold: float = 0.5) -> List[List[str]]:
        """Predict entities for multiple queries."""
        return [self.predict(q, threshold) for q in queries]
    
    def save(self, path: str = './entity_classifier_model'):
        """Save the model."""
        self.model.save_pretrained(path)
        self.tokenizer.save_pretrained(path)
        print(f"Model saved to {path}")
    
    def load(self, path: str = './entity_classifier_model'):
        """Load a saved model."""
        self.tokenizer = AutoTokenizer.from_pretrained(path)
        self.model = AutoModelForSequenceClassification.from_pretrained(path)
        self.model.eval()
        print(f"Model loaded from {path}")

In [ ]:
# Initialize and train classifier
print("=" * 60)
print("APPROACH 2: FINE-TUNED CLASSIFIER (DistilBERT)")
print("=" * 60)

classifier = FineTunedClassifier(num_labels=len(ALL_ENTITIES))
classifier.train(X_train, y_train, epochs=10, batch_size=16)

In [ ]:
# Run predictions on test set
print("\nRunning classifier predictions on test set...")
classifier_predictions = classifier.predict_batch(X_test)

# Encode and evaluate
classifier_pred_encoded = mlb.transform(classifier_predictions)
classifier_metrics = evaluate_predictions(y_test, classifier_pred_encoded, ALL_ENTITIES,
                                          "Fine-tuned Classifier (DistilBERT)")

# Save the model
classifier.save('./entity_classifier_model')

---
# APPROACH 3: RAG (Retrieval-Augmented Generation)

This approach:
- Uses embeddings to find the most similar training examples for each query
- Provides only the TOP-K most relevant examples as context
- Also includes entity descriptions for context
- Uses a pre-trained LLM (no fine-tuning)

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

class RAGEntityPredictor:
    """
    RAG approach: Retrieve relevant examples for each query.
    """
    
    def __init__(self, 
                 llm_name: str = "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
                 embedding_model: str = "all-MiniLM-L6-v2",
                 top_k: int = 10):
        self.llm_name = llm_name
        self.embedding_model_name = embedding_model
        self.top_k = top_k
        self.device = DEVICE
        
        self.llm = None
        self.llm_tokenizer = None
        self.embedding_model = None
        
        self.train_embeddings = None
        self.train_queries = None
        self.train_entities = None
        self.entity_descriptions = None
    
    def load_models(self):
        """Load embedding model and LLM."""
        print(f"Loading embedding model: {self.embedding_model_name}")
        self.embedding_model = SentenceTransformer(self.embedding_model_name)
        
        print(f"Loading LLM: {self.llm_name}")
        self.llm_tokenizer = AutoTokenizer.from_pretrained(self.llm_name)
        
        if self.device == "cuda":
            self.llm = AutoModelForCausalLM.from_pretrained(
                self.llm_name, torch_dtype=torch.float16, device_map="auto"
            )
        else:
            self.llm = AutoModelForCausalLM.from_pretrained(
                self.llm_name, torch_dtype=torch.float32, low_cpu_mem_usage=True
            )
            self.llm.to(self.device)
        
        if self.llm_tokenizer.pad_token is None:
            self.llm_tokenizer.pad_token = self.llm_tokenizer.eos_token
        
        print("Models loaded!")
    
    def index_training_data(self, train_queries: List[str], train_entities: List[List[str]],
                           entity_descriptions: str):
        """Build embedding index for training queries."""
        print(f"Indexing {len(train_queries)} training queries...")
        
        self.train_queries = train_queries
        self.train_entities = train_entities
        self.entity_descriptions = entity_descriptions
        
        # Compute embeddings for all training queries
        self.train_embeddings = self.embedding_model.encode(
            train_queries, show_progress_bar=True, convert_to_numpy=True
        )
        
        print(f"Indexed {len(train_queries)} queries")
    
    def retrieve_similar(self, query: str, k: int = None) -> List[Tuple[str, List[str], float]]:
        """Retrieve top-k most similar training examples."""
        if k is None:
            k = self.top_k
        
        # Compute query embedding
        query_embedding = self.embedding_model.encode([query], convert_to_numpy=True)[0]
        
        # Compute cosine similarities
        similarities = np.dot(self.train_embeddings, query_embedding) / (
            np.linalg.norm(self.train_embeddings, axis=1) * np.linalg.norm(query_embedding)
        )
        
        # Get top-k indices
        top_indices = np.argsort(similarities)[-k:][::-1]
        
        results = []
        for idx in top_indices:
            results.append((
                self.train_queries[idx],
                self.train_entities[idx],
                float(similarities[idx])
            ))
        
        return results
    
    def build_rag_prompt(self, query: str, retrieved_examples: List[Tuple]) -> str:
        """Build prompt with retrieved examples."""
        # Concise entity descriptions
        entity_info = """ENTITY TYPES:
- CDR: Call/SMS/Email records, communications
- Phone: Phone devices, IMEI/IMSI/MSISDN identifiers
- Web Activity: Social media posts, comments, online content
- Web Actor: Social media profiles/accounts
- Person: Individual people with personal info
- Investigation: Cases and inquiries
- Insight: Intelligence analysis notes
- Report: Documentation and reports
- EVisa Request: Visa applications"""
        
        # Build examples section
        examples_text = "\nRELEVANT EXAMPLES:\n"
        for q, entities, sim in retrieved_examples:
            examples_text += f"Query: {q}\nEntities: {entities}\n\n"
        
        prompt = f"""You are an entity extraction system. Given a query, predict which entity types are relevant.

{entity_info}

{examples_text}
Now predict for this query. Return ONLY the entity names as a Python list.
Query: {query}
Entities:"""
        
        return prompt
    
    def predict(self, query: str) -> List[str]:
        """Predict entities using RAG approach."""
        # Retrieve similar examples
        retrieved = self.retrieve_similar(query)
        
        # Build prompt
        prompt = self.build_rag_prompt(query, retrieved)
        
        # Generate response
        inputs = self.llm_tokenizer(prompt, return_tensors="pt", truncation=True, 
                                    max_length=2048)
        inputs = {k: v.to(self.device) for k, v in inputs.items()}
        
        with torch.no_grad():
            outputs = self.llm.generate(
                **inputs,
                max_new_tokens=100,
                do_sample=False,
                pad_token_id=self.llm_tokenizer.pad_token_id,
                eos_token_id=self.llm_tokenizer.eos_token_id,
            )
        
        response = self.llm_tokenizer.decode(outputs[0], skip_special_tokens=True)
        
        # Extract entities from response
        if "Entities:" in response:
            response = response.split("Entities:")[-1].strip()
        
        return parse_llm_response(response, ALL_ENTITIES)
    
    def predict_batch(self, queries: List[str], verbose: bool = True) -> List[List[str]]:
        """Predict entities for multiple queries."""
        predictions = []
        for i, query in enumerate(queries):
            if verbose and i % 20 == 0:
                print(f"Processing {i+1}/{len(queries)}...")
            pred = self.predict(query)
            predictions.append(pred)
        return predictions

In [ ]:
# Initialize RAG predictor
print("=" * 60)
print("APPROACH 3: RAG (Retrieval-Augmented Generation)")
print("=" * 60)

rag_predictor = RAGEntityPredictor(
    llm_name="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    embedding_model="all-MiniLM-L6-v2",
    top_k=10
)
rag_predictor.load_models()
rag_predictor.index_training_data(X_train, train_entities, ENTITY_DESCRIPTIONS)

In [ ]:
# Test retrieval for a sample query
sample_query = "What SMS messages were sent from suspicious phones?"
retrieved = rag_predictor.retrieve_similar(sample_query, k=5)
print(f"\nRetrieved examples for: '{sample_query}'")
for q, entities, sim in retrieved:
    print(f"  [{sim:.3f}] {q[:60]}... -> {entities}")

In [ ]:
# Run predictions on test set
print("\nRunning RAG predictions on test set...")
rag_predictions = rag_predictor.predict_batch(X_test)

# Encode and evaluate
rag_pred_encoded = mlb.transform(rag_predictions)
rag_metrics = evaluate_predictions(y_test, rag_pred_encoded, ALL_ENTITIES,
                                   "RAG (Retrieval-Augmented Generation)")

---
# Results Comparison

In [ ]:
# Compare all approaches
print("\n" + "="*70)
print("COMPARISON OF ALL APPROACHES")
print("="*70)

results_df = pd.DataFrame({
    'Metric': ['Exact Match', 'F1 Micro', 'F1 Macro', 'Precision', 'Recall', 'Jaccard'],
    'Naive LLM': [
        naive_metrics['exact_match'],
        naive_metrics['f1_micro'],
        naive_metrics['f1_macro'],
        naive_metrics['precision_micro'],
        naive_metrics['recall_micro'],
        naive_metrics['jaccard_micro']
    ],
    'Fine-tuned Classifier': [
        classifier_metrics['exact_match'],
        classifier_metrics['f1_micro'],
        classifier_metrics['f1_macro'],
        classifier_metrics['precision_micro'],
        classifier_metrics['recall_micro'],
        classifier_metrics['jaccard_micro']
    ],
    'RAG': [
        rag_metrics['exact_match'],
        rag_metrics['f1_micro'],
        rag_metrics['f1_macro'],
        rag_metrics['precision_micro'],
        rag_metrics['recall_micro'],
        rag_metrics['jaccard_micro']
    ]
})

print(results_df.to_string(index=False))

# Find best approach for each metric
print("\nBest Approach per Metric:")
for idx, row in results_df.iterrows():
    metric = row['Metric']
    values = [row['Naive LLM'], row['Fine-tuned Classifier'], row['RAG']]
    approaches = ['Naive LLM', 'Fine-tuned Classifier', 'RAG']
    best_idx = np.argmax(values)
    print(f"  {metric}: {approaches[best_idx]} ({values[best_idx]:.4f})")

---
# Test Cases

In [ ]:
# Define test cases
test_cases = [
    ("What SMS messages were sent from suspicious phones to 0549876543 containing 'urgent'?", ["CDR", "Phone"]),
    ("Find all calls made using 3G technology", ["CDR"]),
    ("Show me all tweets from accounts with 500 friends mentioning Tesla", ["Web Activity", "Web Actor"]),
    ("Which phones have been marked as suspicious?", ["Phone"]),
    ("Find all individuals with occupation 'engineer' born before July 1985", ["Person"]),
    ("Show me investigations that are open or created in the last 3 months", ["Investigation"]),
    ("Find insights containing 'money laundering' from the past month", ["Insight"]),
    ("List visitors whose travel document was issued before January 2020", ["EVisa Request"]),
    ("Get reports created in the past 3 days", ["Report"]),
    ("Find Instagram profiles with 100 followers using Israel phone number", ["Web Actor"]),
    ("List emails sent to phones associated with target Sarah Johnson", ["CDR", "Phone"]),
]

print("\n" + "="*70)
print("TEST CASES COMPARISON")
print("="*70)

for query, expected in test_cases:
    print(f"\nQuery: {query[:65]}..." if len(query) > 65 else f"\nQuery: {query}")
    print(f"Expected: {expected}")
    
    # Get predictions from each approach
    naive_pred = naive_predictor.predict(query)
    classifier_pred = classifier.predict(query)
    rag_pred = rag_predictor.predict(query)
    
    def check(pred, exp):
        return "✓" if set(pred) == set(exp) else ("~" if set(pred) & set(exp) else "✗")
    
    print(f"  Naive LLM:   {check(naive_pred, expected)} {naive_pred}")
    print(f"  Classifier:  {check(classifier_pred, expected)} {classifier_pred}")
    print(f"  RAG:         {check(rag_pred, expected)} {rag_pred}")

---
# Summary and Recommendations

## Approaches Implemented

### 1. Naive LLM (Full Context)
- **How it works**: Provides ALL training examples + entity descriptions as context to the LLM
- **Pros**: No training required, uses all available information
- **Cons**: Limited by context window, slow inference, may have token truncation
- **Best for**: When you have a small dataset that fits in context

### 2. Fine-tuned Classifier (DistilBERT)
- **How it works**: Trains a BERT model specifically for this multi-label classification task
- **Pros**: Fast inference, high accuracy, learns dataset-specific patterns
- **Cons**: Requires training, fixed label set
- **Best for**: Production deployment with consistent entity types

### 3. RAG (Retrieval-Augmented Generation)
- **How it works**: Retrieves most similar examples for each query, provides them as context
- **Pros**: Scalable, uses relevant context only, no training needed
- **Cons**: Depends on retrieval quality, slower than classifier
- **Best for**: Large datasets, dynamic content, when examples are heterogeneous

## Metrics Explanation
- **Exact Match**: % of queries where ALL predicted entities exactly match ground truth
- **F1 Score**: Harmonic mean of precision and recall (micro=global, macro=per-class average)
- **Precision**: Of predicted entities, how many were correct
- **Recall**: Of actual entities, how many were predicted
- **Jaccard Score**: Intersection over union of predicted and true labels

## Open Issues & Future Improvements

1. **Better LLM**: Use larger models like Phi-2, Mistral-7B, or Llama-2-7B for better reasoning
2. **Ensemble**: Combine predictions from multiple approaches for better accuracy
3. **Threshold Optimization**: Tune classification thresholds per entity type
4. **Data Augmentation**: Generate synthetic queries to improve robustness
5. **Entity Descriptions**: Use field descriptions more effectively in prompts
6. **Hybrid RAG**: Combine embedding retrieval with keyword matching
7. **Active Learning**: Implement feedback loop for continuous improvement

In [ ]:
# Save results
results_df.to_csv('approach_comparison_results.csv', index=False)
print("Results saved to approach_comparison_results.csv")